# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asmajavaid1270/Flyrank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# 1. Load Data (or create a dummy DataFrame if data.csv is not available)
try:
    df = pd.read_csv("data.csv")  # Replace with your dataset path
except FileNotFoundError:
    print("data.csv not found. Creating a dummy DataFrame for demonstration.")
    data = {
        'age': [25, 30, 35, 40, np.nan, 28, 33, 38, 45, 50],
        'income': [50000, 60000, 75000, np.nan, 40000, 55000, 70000, 80000, 90000, 100000],
        'category': ['A', 'B', 'A', 'C', np.nan, 'B', 'A', 'C', 'B', 'A'],
        'created_at': pd.to_datetime(['2022-01-01', '2022-01-05', '2022-01-10', '2022-01-15', '2022-01-20', '2022-01-25', '2022-01-30', '2022-02-01', '2022-02-05', '2022-02-10']),
        'target': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
    }
    df = pd.DataFrame(data)


# 2. Define Features & Target
target_col = "target"
raw_feature_cols = ["age", "income", "category", "created_at"]

# 3. Missing Value Imputation
df["age"] = df["age"].fillna(df["age"].median())
df["income"] = df["income"].fillna(0)
df["category"] = df["category"].fillna("Unknown")

# 4. Categorical Encoding (One-Hot / Frequency Encoding)
encoder = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
cat_encoded = pd.DataFrame(
    encoder.fit_transform(df[["category"]]),
    columns=encoder.get_feature_names_out(["category"]),
)

# 5. Feature Engineering (Only using past/present information)
df["income_per_age"] = df["income"] / (df["age"] + 1)

# Combine into final feature matrix X
X = pd.concat([df[["age", "income", "income_per_age"]], cat_encoded], axis=1)
y = df[target_col]

print("Feature Vector Shape:", X.shape)
print("Features:", list(X.columns))

data.csv not found. Creating a dummy DataFrame for demonstration.
Feature Vector Shape: (10, 6)
Features: ['age', 'income', 'income_per_age', 'category_B', 'category_C', 'category_Unknown']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature Name | Meaning / Description | Missing Value Strategy | Categorical / Numerical | Available BEFORE Prediction? |
| :--- | :--- | :--- | :--- | :--- |
| **age** | Customer age at registration | Imputed with median | Numerical | **Yes** (Collected at signup) |
| **income** | Self-reported annual income | Imputed with 0 | Numerical | **Yes** (Collected at signup) |
| **income_per_age** | Engineered ratio (`income / age`) | Derived from clean columns | Numerical | **Yes** (Computed using historical inputs) |
| **category_B** | One-Hot flag for category B | Filled as `'Unknown'` | Categorical (Binary) | **Yes** (Selection made before prediction) |




In [6]:
import pandas as pd

# Feature notes table structure
data = [
    {
        "Feature Name": "age",
        "Meaning": "Customer age at registration",
        "Missing Value Strategy": "Imputed with median",
        "Type": "Numerical",
        "Available Before Prediction?": "Yes",
    },
    {
        "Feature Name": "income",
        "Meaning": "Self-reported annual income",
        "Missing Value Strategy": "Imputed with 0",
        "Type": "Numerical",
        "Available Before Prediction?": "Yes",
    },
    {
        "Feature Name": "income_per_age",
        "Meaning": "Engineered ratio (income/age)",
        "Missing Value Strategy": "Derived from clean columns",
        "Type": "Numerical",
        "Available Before Prediction?": "Yes",
    },
    {
        "Feature Name": "category_B",
        "Meaning": "One-Hot flag for category B",
        "Missing Value Strategy": "Filled as 'Unknown'",
        "Type": "Categorical (Binary)",
        "Available Before Prediction?": "Yes",
    },
]

feature_notes_df = pd.DataFrame(data)
display(feature_notes_df)

,Feature Name,Meaning,Missing Value Strategy,Type,Available Before Prediction?
0,age,Customer age at registration,Imputed with median,Numerical,Yes
1,income,Self-reported annual income,Imputed with 0,Numerical,Yes
2,income_per_age,Engineered ratio (income/age),Derived from clean columns,Numerical,Yes
3,category_B,One-Hot flag for category B,Filled as 'Unknown',Categorical (Binary),Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# 1. Check high correlation with target (Flag > 0.95 correlation as potential leakage)
correlations = X.apply(lambda col: col.corr(y)).abs()
potential_leakage = correlations[correlations > 0.95]

print("--- Data Leakage Test ---")
if not potential_leakage.empty:
    print(
        "⚠️ WARNING: High correlation detected with target! Check columns:",
        potential_leakage,
    )
else:
    print("✅ No direct linear correlation leakage found.")

# 2. Time-based Leakage Check (Ensure feature timestamp < prediction timestamp)
# Example: feature_time <= event_time
if "feature_timestamp" in df.columns and "event_timestamp" in df.columns:
    future_leakage_count = (
        df["feature_timestamp"] > df["event_timestamp"]
    ).sum()
    print(f"Future feature timestamps found: {future_leakage_count}")

--- Data Leakage Test ---
✅ No direct linear correlation leakage found.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


**post_signup_purchases:** Excluded because this field occurs after the prediction timestamp (Future leakage).

**user_ssn / full_name:** Excluded to comply with privacy/PII policy and avoid standard bias.

**churn_status_raw:** Excluded because it is directly derived from the target variable (label leakage).

**ip_address:** Excluded due to high cardinality and potential location privacy restrictions.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.